In [ ]:
%load_ext autoreload
%autoreload 2

**Author:** Salvador Navas  
**Date:** 2025-06-27

In [ ]:
from pyhydra.climate.stochastic_generation import (
    SpatialFieldModel,
    fit_spatial_model,
    generate_random_field,
    generate_random_field_fast,
    check_random_field,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Spatial Random Field Generation (CoSMoS_py)

Generates spatially (and spatio-temporally) correlated random fields that
preserve a target marginal distribution while reproducing a user-specified
spatio-temporal correlation structure (STCS).

**Use cases:**
- Gridded daily precipitation ensembles for hydrological modelling
- Spatially correlated temperature or wind fields
- Stochastic weather generators that account for spatial dependence

**Methodology:** VAR(p) model with ACTF (Biller-Nelson 2003) — a copula-based
mapping from Gaussian AR residuals to the target marginal.

```bash
pip install -e /path/to/CoSMoS_py
# Source: https://github.com/navass11/CoSMoS_py
```

| Parameter | Options | Description |
|-----------|---------|-------------|
| `dist` | `gengamma`, `burrXII`, `paretoII`, `gev`, `lnorm` | Marginal distribution |
| `stcs_id` | `clayton`, `gneiting14`, `gneiting16` | Space-time correlation |
| `dep_structure` | `gauss`, `student`, `bardossy` | Spatial copula |
| `advection_id` | `uniform`, `rotation`, `spiral` | Storm advection pattern |

---
## 1. Regular grid — daily precipitation

Simulate 1 year of daily precipitation on a 10×10 grid (100 pixels).
The Clayton copula STCS combines independent spatial and temporal
ACFs via a Clayton copula — flexible and parsimonious.

In [ ]:
# Marginal: generalised gamma (common choice for precipitation)
# scale=5, shape1=0.8, shape2=0.5 → mean ≈ 5 mm, moderate variability
# p0=0.65 → 65% dry days

model_precip = SpatialFieldModel(
    dist="gengamma",
    dist_params={"scale": 5.0, "shape1": 0.8, "shape2": 0.5},
    p0=0.65,
    p=1,
    stcs_id="clayton",
    stcs_params={
        "scfid":     "weibull",
        "tcfid":     "weibull",
        "copulaarg": 2.0,           # θ > 0 → stronger space-time coupling
        "scfarg":    {"b": 0.3, "c": 0.5},   # spatial ACF decay
        "tcfarg":    {"b": 0.5, "c": 0.8},   # temporal ACF decay
    },
)

# Build VAR model for a 10×10 regular grid
# (requires CoSMoS_py installed)
# model_precip.fit(spacepoints=10)
print("(Building the VAR model requires CoSMoS_py)")
print("  model_precip.fit(spacepoints=10)")

In [ ]:
# Simulate 365 days
# field = model_precip.simulate(n_steps=365)   # shape (365, 100)
# print(f"Simulated field: {field.shape}")
# print(f"Mean daily precipitation: {field.mean():.2f} mm")
# print(f"Wet-day fraction: {(field > 0).mean():.3f}")

print("(Simulation requires fit() to be run first)")
print("  field = model_precip.simulate(n_steps=365)")

In [ ]:
# Visualise 4 consecutive days
# m = 10   # grid side
# fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
# for i, (ax, day_idx) in enumerate(zip(axes, [0, 30, 90, 180])):
#     Z = field[day_idx].reshape(m, m)
#     im = ax.imshow(Z, cmap="Blues", vmin=0, vmax=field.max())
#     ax.set_title(f"Day {day_idx + 1}", fontsize=10)
#     plt.colorbar(im, ax=ax, label="mm" if i == 3 else "")
# plt.suptitle("Simulated daily precipitation — 10×10 grid", fontsize=12)
# plt.tight_layout()
# plt.show()

print("(Visualisation — uncomment after simulation)")

---
## 2. Irregular station locations

Pass a (n_sites × 2) coordinate array to simulate at arbitrary locations —
useful for validating synthetic series against gauging stations.

In [ ]:
# 8 gauge stations (lon/lat or any Euclidean coordinates in km)
coords = np.array([
    [0.0, 0.0],
    [5.0, 0.0],
    [10.0, 0.0],
    [2.5, 4.0],
    [7.5, 4.0],
    [0.0, 8.0],
    [5.0, 8.0],
    [10.0, 8.0],
])

model_stations = SpatialFieldModel(
    dist="gengamma",
    dist_params={"scale": 5.0, "shape1": 0.8, "shape2": 0.5},
    p0=0.65,
    stcs_id="clayton",
)

# model_stations.fit(spacepoints=coords)
# field_stations = model_stations.simulate(n_steps=3650)   # 10 years
# print(f"Simulated at {coords.shape[0]} sites over 10 years: {field_stations.shape}")

print("(Irregular site simulation — requires CoSMoS_py)")
print("  model_stations.fit(spacepoints=coords)")
print("  field_stations = model_stations.simulate(n_steps=3650)")

In [ ]:
# Plot station positions and simulated mean precipitation
# mean_by_site = field_stations.mean(axis=0)

# fig, ax = plt.subplots(figsize=(6, 6))
# sc = ax.scatter(coords[:, 0], coords[:, 1], c=mean_by_site,
#                 cmap="YlGnBu", s=200, edgecolors="k", linewidths=0.8)
# plt.colorbar(sc, ax=ax, label="Mean daily precip (mm)")
# for i, (x, y) in enumerate(coords):
#     ax.annotate(f"S{i}", (x, y), textcoords="offset points",
#                 xytext=(6, 4), fontsize=9)
# ax.set_xlabel("X (km)"); ax.set_ylabel("Y (km)")
# ax.set_title("Mean daily precipitation — 8 stations", fontsize=11)
# ax.set_aspect("equal")
# ax.grid(alpha=0.3)
# plt.tight_layout()
# plt.show()

print("(Site plot — uncomment after simulation)")

---
## 3. Spatio-temporal correlation structures

The STCS controls how correlation decays with both distance and time lag.
Three structures are available:

| ID | Model | Key parameters |
|----|-------|----------------|
| `clayton` | Clayton copula of spatial and temporal ACFs | `copulaarg` (θ), `scfarg`, `tcfarg` |
| `gneiting14` | Gneiting (2002) Eq.14 | `a`, `c`, `alpha`, `beta`, `gamma`, `tau` |
| `gneiting16` | Gneiting (2002) Eq.16 (Matérn) | `a`, `c`, `alpha`, `beta`, `nu`, `tau` |

In [ ]:
# Visualise STCS decay curves
try:
    from cosmos_py.fields.stcs import stcfclayton, stcfgneiting14, stcfgneiting16
    from cosmos_py.correlation.acs import get_acs

    s_range = np.linspace(0, 10, 100)   # spatial distances
    t_range = np.linspace(0, 10, 100)   # time lags

    # Clayton: spatial decay at t=0 (purely spatial slice)
    clay_s = stcfclayton(
        t=np.zeros_like(s_range), s=s_range,
        scfid="weibull", tcfid="weibull",
        copulaarg=2.0,
        scfarg={"b": 0.3, "c": 0.5},
        tcfarg={"b": 0.5, "c": 0.8},
    )

    # Gneiting14: spatial slice at t=0
    gnei14_s = stcfgneiting14(
        t=np.zeros_like(s_range), s=s_range,
        a=1.0, c=1.0, alpha=0.5, beta=0.5, gamma=0.5, tau=1.0,
    )

    # Gneiting16 (Matérn)
    gnei16_s = stcfgneiting16(
        t=np.zeros_like(s_range), s=s_range,
        a=1.0, c=1.0, alpha=0.5, beta=0.5, nu=1.5, tau=1.0,
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Spatial decay
    axes[0].plot(s_range, clay_s,   label="Clayton")
    axes[0].plot(s_range, gnei14_s, label="Gneiting14")
    axes[0].plot(s_range, gnei16_s, label="Gneiting16 (Matérn)")
    axes[0].set_xlabel("Spatial distance")
    axes[0].set_ylabel("Correlation")
    axes[0].set_title("Spatial decay at t=0", fontsize=11)
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Temporal decay of Clayton at s=0
    clay_t = stcfclayton(
        t=t_range, s=np.zeros_like(t_range),
        scfid="weibull", tcfid="weibull",
        copulaarg=2.0,
        scfarg={"b": 0.3, "c": 0.5},
        tcfarg={"b": 0.5, "c": 0.8},
    )
    gnei14_t = stcfgneiting14(
        t=t_range, s=np.zeros_like(t_range),
        a=1.0, c=1.0, alpha=0.5, beta=0.5, gamma=0.5, tau=1.0,
    )

    axes[1].plot(t_range, clay_t,   label="Clayton")
    axes[1].plot(t_range, gnei14_t, label="Gneiting14")
    axes[1].set_xlabel("Time lag")
    axes[1].set_ylabel("Correlation")
    axes[1].set_title("Temporal decay at s=0", fontsize=11)
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle("STCS comparison", fontsize=12)
    plt.tight_layout()
    plt.show()

except ImportError:
    print("(CoSMoS_py not installed — pip install -e /path/to/CoSMoS_py)")

---
## 4. Spatial copula (dependence structure)

The `dep_structure` parameter selects the copula type used for the spatial
residuals in the VAR model:

| `dep_structure` | Tails | When to use |
|-----------------|-------|-------------|
| `gauss` | Light (symmetric) | Default; most cases |
| `student` | Heavy (symmetric) | Concurrent extremes important |
| `bardossy` | Asymmetric | Asymmetric spatial extremes |

In [ ]:
# Student-t copula — heavier joint tails
model_student = SpatialFieldModel(
    dist="gengamma",
    dist_params={"scale": 5.0, "shape1": 0.8, "shape2": 0.5},
    p0=0.65,
    dep_structure="student",
    dep_arg=5.0,   # degrees of freedom — lower = heavier tails
)
# model_student.fit(spacepoints=10)
# field_student = model_student.simulate(n_steps=365)

# Bardossy copula
model_bardossy = SpatialFieldModel(
    dist="gengamma",
    dist_params={"scale": 5.0, "shape1": 0.8, "shape2": 0.5},
    p0=0.65,
    dep_structure="bardossy",
    dep_arg=2.0,
)
# model_bardossy.fit(spacepoints=10)

print("Student-t model:", model_student.dep_structure, "df =", model_student.dep_arg)
print("Bardossy model :", model_bardossy.dep_structure, "m  =", model_bardossy.dep_arg)

---
## 5. Advection

Advection shifts the field in space between time steps — simulating storm
movement. Uniform advection (wind vector) is the most common choice.

In [ ]:
# Westerly advection at 2 km/h
model_adv = SpatialFieldModel(
    dist="gengamma",
    dist_params={"scale": 5.0, "shape1": 0.8, "shape2": 0.5},
    p0=0.65,
    advection_id="uniform",
    advection_params={"u": 2.0, "v": 0.0},   # u = East-West, v = North-South
)
# model_adv.fit(spacepoints=15)
# field_adv = model_adv.simulate(n_steps=365)

# Available advection types:
# 'uniform'  — constant wind vector
# 'rotation' — rotating storm
# 'spiral'   — spiral inward/outward
# 'radial'   — radial divergence/convergence
# 'hyperbolic' — saddle-point flow

print("Advection model:", model_adv.advection_id, model_adv.advection_params)

---
## 6. DataFrame output and simulation period

`simulate_dataframe` wraps the raw ndarray into a pandas DataFrame with a
DatetimeIndex — ready for analysis or export.

In [ ]:
# model_precip.fit(spacepoints=10)
# df_field = model_precip.simulate_dataframe(
#     n_steps=3650,
#     start_date="1990-01-01",
#     freq="D",
# )
# print(df_field.shape)         # (3650, 100)
# print(df_field.head(2))

# Annual precipitation at each grid cell
# annual = df_field.resample("YE").sum()
# print("Annual precipitation range:", annual.values.min().round(0), "–", annual.values.max().round(0), "mm")

print("(DataFrame simulation — uncomment after fit())")

---
## 7. Diagnostics

`check_random_field` compares simulated statistics against the target model
to verify that the VAR fitting and simulation converged correctly.

In [ ]:
# Run a quick 1000-step simulation and check statistics
# diag = model_precip.diagnostics(n_steps=1000)

# print("Marginal statistics:")
# print(diag['marginal'].round(3))

# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# axes[0].plot(diag['spatial_acf'], 'b-o', ms=4, label='Simulated')
# axes[0].set_title('Spatial ACF'); axes[0].set_xlabel('Distance lag')
# axes[0].set_ylabel('Correlation'); axes[0].grid(alpha=0.3)

# axes[1].plot(diag['temporal_acf'], 'r-o', ms=4, label='Simulated')
# axes[1].set_title('Temporal ACF'); axes[1].set_xlabel('Time lag')
# axes[1].set_ylabel('Correlation'); axes[1].grid(alpha=0.3)

# plt.suptitle('Diagnostics — simulated vs target', fontsize=12)
# plt.tight_layout()
# plt.show()

print("(Diagnostics — uncomment after fit())")

---
## 8. Temperature field — continuous variable (no zeros)

For temperature or wind speed, set `p0=0` and choose a distribution that
supports the full real line (e.g. Normal) or positive values (LogNormal).

In [ ]:
# Daily mean temperature — LogNormal marginal (shifted), no zeros
model_temp = SpatialFieldModel(
    dist="lnorm",
    dist_params={"meanlog": 2.5, "sdlog": 0.3},   # mean ≈ 13°C
    p0=0.0,                                        # no zeros
    p=2,                                           # VAR(2) for longer memory
    stcs_id="gneiting14",
    stcs_params={"a": 0.5, "c": 0.8, "alpha": 0.7,
                 "beta": 0.3, "gamma": 0.8, "tau": 1.0},
)
# model_temp.fit(spacepoints=10)
# field_temp = model_temp.simulate(n_steps=365)

print("Temperature model:", model_temp.dist, "p0 =", model_temp.p0)

---
## 9. Functional API

The same steps are available as standalone functions for scripting
or integration into larger pipelines.

In [ ]:
# Functional API equivalent of sections 1-2

# var_model = fit_spatial_model(
#     spacepoints=10,
#     p=1,
#     dist="gengamma",
#     dist_params={"scale": 5.0, "shape1": 0.8, "shape2": 0.5},
#     p0=0.65,
#     stcs_id="clayton",
# )

# field = generate_random_field(365, var_model)
# field_fast = generate_random_field_fast(3650, var_model)  # faster for long simulations

# diagnostics = check_random_field(field, var_model)

print("Functional API:")
print("  var_model = fit_spatial_model(spacepoints=10, p=1, dist='gengamma', ...)")
print("  field     = generate_random_field(365, var_model)")
print("  diag      = check_random_field(field, var_model)")